In [1]:
!curl -o hand_landmarker.task https://storage.googleapis.com/mediapipe-models/hand_landmarker/hand_landmarker/float16/1/hand_landmarker.task

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed

  0     0    0     0    0     0      0      0 --:--:-- --:--:-- --:--:--     0
 10 7635k   10  837k    0     0  1426k      0  0:00:05 --:--:--  0:00:05 1428k
100 7635k  100 7635k    0     0  5781k      0  0:00:01  0:00:01 --:--:-- 5789k


In [2]:
import cv2
import mediapipe as mp
from mediapipe.tasks import python
from mediapipe.tasks.python import vision
import numpy as np
import math
from pycaw.pycaw import AudioUtilities

# Initialize PyCaw
try:
    devices = AudioUtilities.GetSpeakers()
    volume = devices.EndpointVolume
    vol_range = volume.GetVolumeRange()
    min_vol, max_vol = vol_range[0], vol_range[1]
except Exception:
    from comtypes import CLSCTX_ALL
    from pycaw.pycaw import IAudioEndpointVolume
    interface = devices.Activate(IAudioEndpointVolume._iid_, CLSCTX_ALL, None)
    volume = interface.QueryInterface(IAudioEndpointVolume)
    vol_range = volume.GetVolumeRange()
    min_vol, max_vol = vol_range[0], vol_range[1]

vol_bar = 400
vol_per = 0

HAND_CONNECTIONS = [
    (0, 1), (1, 2), (2, 3), (3, 4), (0, 5), (5, 6), (6, 7), (7, 8),
    (0, 9), (9, 10), (10, 11), (11, 12), (0, 13), (13, 14), (14, 15), (15, 16),
    (0, 17), (17, 18), (18, 19), (19, 20), (5, 9), (9, 13), (13, 17), (0, 17)
]

base_options = python.BaseOptions(model_asset_path="hand_landmarker.task")
options = vision.HandLandmarkerOptions(
    base_options=base_options,
    num_hands=1,
    min_hand_detection_confidence=0.7,
    running_mode=vision.RunningMode.IMAGE
)
print("Initialization successful. Ready for the camera loop.")

Initialization successful. Ready for the camera loop.


In [3]:
# Open Webcam
cap = cv2.VideoCapture(0)

cv2.startWindowThread()
cv2.namedWindow("Gesture Volume Control", cv2.WINDOW_AUTOSIZE)
cv2.setWindowProperty("Gesture Volume Control",
                      cv2.WND_PROP_TOPMOST,
                      1)

print("Loop running! Press 'q' or 'Esc' to exit.")

with vision.HandLandmarker.create_from_options(options) as landmarker:

    while True:

        success, img = cap.read()

        if not success:
            continue

        img = cv2.flip(img, 1)
        h, w, _ = img.shape

        # -----------------------------
        # Glass Panel
        # -----------------------------
        overlay = img.copy()
        cv2.rectangle(overlay, (10, 10), (350, 80), (35, 35, 35), -1)
        cv2.addWeighted(overlay, 0.45, img, 0.55, 0, img)

        img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        mp_image = mp.Image(image_format=mp.ImageFormat.SRGB,
                            data=img_rgb)

        result = landmarker.detect(mp_image)

        if result.hand_landmarks:

            for hand_landmarks in result.hand_landmarks:

                pts = [(int(lm.x * w), int(lm.y * h))
                       for lm in hand_landmarks]

                # White Hand Skeleton
                for start, end in HAND_CONNECTIONS:
                    cv2.line(img,
                             pts[start],
                             pts[end],
                             (255, 255, 255),
                             2)

                # Orange Landmarks
                for pt in pts:
                    cv2.circle(img,
                               pt,
                               5,
                               (0, 165, 255),
                               cv2.FILLED)

                x1, y1 = pts[4]
                x2, y2 = pts[8]

                # Pink Thumb & Index
                cv2.circle(img,
                           (x1, y1),
                           12,
                           (255, 0, 255),
                           cv2.FILLED)

                cv2.circle(img,
                           (x2, y2),
                           12,
                           (255, 0, 255),
                           cv2.FILLED)

                cv2.line(img,
                         (x1, y1),
                         (x2, y2),
                         (255, 0, 255),
                         4)

                length = math.hypot(x2 - x1, y2 - y1)

                vol = np.interp(length,
                                [20, 200],
                                [min_vol, max_vol])

                vol_bar = np.interp(length,
                                    [20, 200],
                                    [400, 150])

                vol_per = np.interp(length,
                                    [20, 200],
                                    [0, 100])

                volume.SetMasterVolumeLevel(vol, None)

        # -----------------------------
        # Gradient Volume Bar
        # -----------------------------

        if vol_per < 35:
            bar_color = (0, 255, 0)

        elif vol_per < 70:
            bar_color = (0, 255, 255)

        else:
            bar_color = (0, 0, 255)

        cv2.rectangle(img,
                      (50, 150),
                      (85, 400),
                      (255, 255, 255),
                      2)

        cv2.rectangle(img,
                      (50, int(vol_bar)),
                      (85, 400),
                      bar_color,
                      cv2.FILLED)

        # -----------------------------
        # Information Panel
        # -----------------------------

        cv2.putText(img,
                    "Gesture Volume Control",
                    (20, 40),
                    cv2.FONT_HERSHEY_DUPLEX,
                    0.8,
                    (255, 255, 255),
                    2)

        cv2.putText(img,
                    f"Volume : {int(vol_per)}%",
                    (20, 70),
                    cv2.FONT_HERSHEY_SIMPLEX,
                    0.65,
                    (255, 255, 255),
                    2)

        cv2.putText(img,
                    "Move Thumb & Index Finger",
                    (120, 430),
                    cv2.FONT_HERSHEY_SIMPLEX,
                    0.6,
                    (255, 255, 255),
                    2)

        cv2.imshow("Gesture Volume Control", img)

        key = cv2.waitKey(1) & 0xFF

        # q OR Esc OR Window Close Button
        if (
            key == ord("q")
            or key == 27
            or cv2.getWindowProperty(
                "Gesture Volume Control",
                cv2.WND_PROP_VISIBLE
            ) < 1
        ):
            break

cap.release()
cv2.destroyAllWindows()

print("Camera closed successfully.")

Loop running! Press 'q' or 'Esc' to exit.
Camera closed successfully.
